# 3.3 tiling 分核

## 概述

上一节实现了 ReLU，并发现 shape=(65536,)输入若一次全部放入单核 UB，输入与结果共需 512 KiB，超过当前环境可分配的UB容量（248 KiB）。本节先将数据尽量均分到多个核，再检查每个核的数据是否放得下；若仍超过 UB 容量，就在该核负责的区间内分块，循环完成读入、计算与写回。我们先用 shape=(65536,)说明分核，再用 shape=(524288,)说明核内分块，最后验证完整输出并比较 kernel 耗时。

## 学习目标

- 将输入按连续区间分配给多个核，计算每个核负责的数据量。
- 根据 UB 空间确定核内 tile 大小，说明为什么需要循环分块。
- 推导核起点与轮次偏移，保证读写索引覆盖完整、不重不漏。
- 验证 ReLU 的分核与分块实现，并将相同方法用于 Exp 练习。

[本章导航](README.md) · [上一节](03.02_single_vector.ipynb) · [下一节](03.04_pipeline.ipynb)

<p><small>运行前完成第 1.4 节的环境准备，并使用本章 README 指定的已验证运行入口启动新内核。每页独立执行，工作目录为本章根目录；内核本身不初始化 NPU。</small></p>

## 1. 先分核：将输入均分给各核

先回答“每个核负责哪些数据”。为了便于说明，这里的示例采用 8 个核进行讲解，用 `blocks=8` 表示，用 `bx` 表示核任务编号。先考虑 N 能被 blocks 整除的情况：每个核负责 `per_core = N // blocks` 项，起点为 `core_begin = bx * per_core`。

回到 N=65536：分给 8 个核后，每核只有 `65536 / 8 = 8192` 项。输入 a 和结果 c 各占 32 KiB，合计 64 KiB，未超过当前环境可分配的UB容量（248 KiB）。因此，这个用例仅通过分核就能解决容量问题，不必为了放入 UB 再切小块。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">核任务</th>
<th style="text-align: left; vertical-align: top;">负责的元素区间（左闭右开）</th>
<th style="text-align: left; vertical-align: top;">元素数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">core0</td>
<td style="text-align: left; vertical-align: top;">[0, 8192)</td>
<td style="text-align: left; vertical-align: top;">8192</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">core1</td>
<td style="text-align: left; vertical-align: top;">[8192, 16384)</td>
<td style="text-align: left; vertical-align: top;">8192</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">…</td>
<td style="text-align: left; vertical-align: top;">…</td>
<td style="text-align: left; vertical-align: top;">…</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">core7</td>
<td style="text-align: left; vertical-align: top;">[57344, 65536)</td>
<td style="text-align: left; vertical-align: top;">8192</td>
</tr>
</tbody>
</table>

各核之间没有先后关系，每个核使用独立的 UB 空间。

**教学配置说明**：本节仅为了方便说明原理，不一定采用最优的 tiling 方式。实际 NPU 的向量核数依型号和配置可能为 64 或 72，当前目标平台配置报告为 72 个向量核，本例为了便于讲解只使用 8 个核任务；`blocks=8` 是本例的配置，不代表设备只有 8 个物理核。实际开发还需结合目标设备、数据规模和实测性能选择核数与 tile。

## 2. 再分块：在每个核内复用 UB

现在把输入扩大为 N=524288，仍分给 8 个核。每核需要处理 65536 项；如果在核内一次读入全部数据，两份 fp32 内存仍需 512 KiB，超过 UB 容量。**分核之后，还要检查每个核自己的空间需求。**

在每个核负责的连续区间内，再按 tile=8192 分块。每次只读入 8192 项，计算并写回后复用同样的 UB 内存，处理下一块。每核循环 `65536 / 8192 = 8` 轮，a、c 合计始终占 64 KiB。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">输入 N</th>
<th style="text-align: left; vertical-align: top;">核任务数</th>
<th style="text-align: left; vertical-align: top;">每核元素数</th>
<th style="text-align: left; vertical-align: top;">每次 tile</th>
<th style="text-align: left; vertical-align: top;">每核轮数</th>
<th style="text-align: left; vertical-align: top;">a、c 合计 UB</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">65536</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">64 KiB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">524288</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">65536</td>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">64 KiB</td>
</tr>
</tbody>
</table>

这里完整输入 A、输出 C 的 shape 为 `(N,)`，核内 a、c 的 shape 为 `(tile,)`。核负责的总数据量可以大于 tile，但 UB 不需要同时保存该核的所有 tile。

![先分核，再在核内分块](images/03.03_tiling.png)

图中先把 524288 项输入均分成 8 个连续区间，再展开 core0：它负责的 65536 项分成 8 个 tile，每个 tile 8192 项。各核并列，core0 内的 tile 按轮次处理；每轮读入和写回均使用该轮的同一索引区间。

## 3. 算子实现与索引计算

将“核负责的区间”和“核内第几块”分别写成两个偏移：

- 核起点：`core_begin = bx * (N // blocks)`。
- 核内偏移：`tile_offset = it * tile`。
- 本轮起点：`begin = core_begin + tile_offset`，读写区间为 `[begin, begin + tile)`。

下面为核函数内部的索引与计算片段，需放在 `@T.prim_func` 中；完整可执行入口为 `python src/run_vector.py tiling`。

```python
with T.Kernel(blocks) as bx:
    a = T.alloc_shared((tile,), "float32")
    c = T.alloc_shared((tile,), "float32")

    for it in T.serial(N // (blocks * tile)):
        begin = bx * (N // blocks) + it * tile
        T.copy(A[begin : begin + tile], a)

        with T.SimdVF():
            mask = T.simd.pset(32)
            zero = T.simd.vdup(0.0, "float32", mask)
            for group in T.serial(tile // 64):
                ra = T.simd.vld(a[group * 64])
                rc = T.simd.vmax(ra, zero, mask)
                T.simd.vsts(c[group * 64], rc, mask)

        T.copy(c, C[begin : begin + tile])
```

以 N=524288、blocks=8、tile=8192 为例：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">bx</th>
<th style="text-align: left; vertical-align: top;">it</th>
<th style="text-align: left; vertical-align: top;">核起点</th>
<th style="text-align: left; vertical-align: top;">核内偏移</th>
<th style="text-align: left; vertical-align: top;">读写区间</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">[0, 8192)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">7</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">57344</td>
<td style="text-align: left; vertical-align: top;">[57344, 65536)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">65536</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">[65536, 73728)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">7</td>
<td style="text-align: left; vertical-align: top;">7</td>
<td style="text-align: left; vertical-align: top;">458752</td>
<td style="text-align: left; vertical-align: top;">57344</td>
<td style="text-align: left; vertical-align: top;">[516096, 524288)</td>
</tr>
</tbody>
</table>

每核的各块首尾相接，各核的区间也首尾相接；最后一块右端点等于 N。因此既不会漏算，也不会重复写入。

### 补充说明：当前示例暂不处理尾块

尾块是分块后剩余数据不足一个完整 tile 的最后一块。例如每核有 1000 个元素、tile=256，前三块各有 256 个元素，最后剩余 232 个元素，这一块就是尾块。
本节选择正整数参数，并约定 `N % (blocks * tile) == 0`，让各核任务等长、每轮都是完整 tile；教学入口还要求 tile 是 64 的倍数。**整除是当前简化写法的约定，不是 tiling 分核在所有实现中都必须满足的条件。** UB 容量则是实际需要遵守的约束：本例要求 `2 * tile * 4 <= 248 * 1024`。

如果输入不能均分，可以让部分核多处理一些元素；如果核内最后一块不足 tile，则需要计算有效长度，并在读入、计算、写回时处理边界。



完整实现见 [vector_kernels.py](src/vector_kernels.py) 的 `unary_serial`。

## 4. 验证分核与分块的结果

按前面的两步选择用例：先验证 65536 项分给 8 核、每核一次完成；再验证 524288 项分给 8 核、每核分 8 轮。两者都检查完整输出。

输入为连续、有限的 fp32，seed=20260911，覆盖随机、零、常数和跨零线性序列。ReLU 与 `torch.relu` 精确比较，检查每个完整输出。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">用例</th>
<th style="text-align: left; vertical-align: top;">N</th>
<th style="text-align: left; vertical-align: top;">tile</th>
<th style="text-align: left; vertical-align: top;">blocks</th>
<th style="text-align: left; vertical-align: top;">每核轮数</th>
<th style="text-align: left; vertical-align: top;">验证目的</th>
<th style="text-align: left; vertical-align: top;">结果</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">先分核</td>
<td style="text-align: left; vertical-align: top;">65536</td>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">均分后单核 UB 足够</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">再在核内分块</td>
<td style="text-align: left; vertical-align: top;">524288</td>
<td style="text-align: left; vertical-align: top;">8192</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">大输入复用固定 UB</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过</td>
</tr>
</tbody>
</table>



下面的执行单元验证分块算子的正确性；性能表随后列出。

In [1]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)
# 独立子进程使用当前内核的解释器；非零返回码会中止本页。
command = [sys.executable, 'src/run_vector.py' , 'tiling']
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as error:
    print(error.stdout)
    print(error.stderr)
    raise
log_name = Path(command[1]).stem + ("_" + command[2] if len(command)>2 else "") + "_notebook.log"
Path(OUTPUT_DIR / "validation", log_name).write_text(result.stdout + result.stderr)
# 输出节选：省略编译日志与安装路径；完整运行日志写入临时输出目录。
print("\n".join(line for line in result.stdout.splitlines()
                if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

ReLU N=65536 tile=8192 blocks=8 random: full-output PASS; shape=(65536,); max_abs_error=0
ReLU N=65536 tile=8192 blocks=8 zero: full-output PASS; shape=(65536,); max_abs_error=0
ReLU N=65536 tile=8192 blocks=8 constant: full-output PASS; shape=(65536,); max_abs_error=0
ReLU N=65536 tile=8192 blocks=8 ramp: full-output PASS; shape=(65536,); max_abs_error=0
ReLU N=524288 tile=8192 blocks=8 random: full-output PASS; shape=(524288,); max_abs_error=0
ReLU N=524288 tile=8192 blocks=8 zero: full-output PASS; shape=(524288,); max_abs_error=0
ReLU N=524288 tile=8192 blocks=8 constant: full-output PASS; shape=(524288,); max_abs_error=0
ReLU N=524288 tile=8192 blocks=8 ramp: full-output PASS; shape=(524288,); max_abs_error=0
ReLU N=1179648 tile=1024 blocks=1 random: full-output PASS; shape=(1179648,); max_abs_error=0
ReLU N=1179648 tile=1024 blocks=1 zero: full-output PASS; shape=(1179648,); max_abs_error=0
ReLU N=1179648 tile=1024 blocks=1 constant: full-output PASS; shape=(1179648,); max_abs_er

### 用 msprof 比较 kernel 耗时

<p><small>工具说明：msprof 是昇腾的性能采集工具，用于获取算子在设备上的执行耗时等信息。其详细用法和性能分析方法将在第6章系统介绍。本章先按给定命令完成采集，读取 kernel 耗时并比较优化前后的结果，无需提前掌握全部参数。</small></p>

为了与 3.4 形成连续对比，性能实验统一使用 ReLU、`shape=(1179648,)`、`tile=1024`。本节先只改变 `blocks=1/8`，下一节保留相同输入和 tile，并固定 `blocks=8`，进一步比较多 buffer。

一核时循环 `1179648/1024=1152` 轮；分给 8 核后，每核负责 147456 项，循环 `147456/1024=144` 轮。单次 a、c 合计占 8 KiB UB。前面的 `(65536,)` 与 `(524288,)` 用于讲解容量和索引；性能表使用这里统一的较大用例。

两组使用相同的连续 fp32 输入，先确认完整输出正确，再用 msprof 采集设备执行耗时。在 op_summary 中筛选 `Op Name=main_kernel`、`Task Type=AI_VECTOR_CORE` 和预期 `Block Num=1/8`，读取 `Task Duration(us)`；不计入 Task Wait Time、Python 调用和 Host 同步等待。

每组先执行 1 次完整精度检查，随后采集 3 批，每批预热 10 次、保留 30 次，共 90 个有效样本。复用同一输入、不清缓存，每次调用前后同步 NPU；每批结束还检查完整输出。

```bash
source src/env.sh
msprof --output="${COURSE_OUTPUT_DIR}/profiles/msprof_blocks1" --task-time=on --ai-core=off --runtime-api=on python src/profile_tiling.py 1
msprof --output="${COURSE_OUTPUT_DIR}/profiles/msprof_blocks8" --task-time=on --ai-core=off --runtime-api=on python src/profile_tiling.py 8
```



[采集程序](src/profile_tiling.py)



### kernel 耗时对比

统一性能用例：`shape=(1179648,)`、`tile=1024`、ReLU、fp32。以下数据为此前寄存器微指令实现的测量记录。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">blocks</th>
<th style="text-align: left; vertical-align: top;">每核轮数</th>
<th style="text-align: left; vertical-align: top;">第1批中位数/μs</th>
<th style="text-align: left; vertical-align: top;">第2批中位数/μs</th>
<th style="text-align: left; vertical-align: top;">第3批中位数/μs</th>
<th style="text-align: left; vertical-align: top;">90次样本中位数/μs</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">1152</td>
<td style="text-align: left; vertical-align: top;">238.6865</td>
<td style="text-align: left; vertical-align: top;">238.6545</td>
<td style="text-align: left; vertical-align: top;">238.5765</td>
<td style="text-align: left; vertical-align: top;">238.6370</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">8</td>
<td style="text-align: left; vertical-align: top;">144</td>
<td style="text-align: left; vertical-align: top;">31.7430</td>
<td style="text-align: left; vertical-align: top;">31.7160</td>
<td style="text-align: left; vertical-align: top;">31.6875</td>
<td style="text-align: left; vertical-align: top;">31.7215</td>
</tr>
</tbody>
</table>

课程参考测量中，8 核相对 1 核加速 **7.52 倍**。每个任务的串行轮数从 1152 降到 144，输入与 tile 均未改变。下一节直接使用这里的 8 核串行结果 **31.7215 μs** 作为多 buffer 对比基线。

## 小结

先将输入分配到各核，再根据每个核的 UB 空间决定一次处理多少数据；索引由核起点和核内偏移组成。65536 项分给 8 核后可一次完成每核计算；524288 项则需要在每核内继续分块。下一节在多轮循环的基础上增加多 buffer，使搬运与计算有机会交叠。

> 表中性能为课程参考测量记录，原始采集附件不在课程目录中。可按本节给出的命令重新采集当前环境的结果。

## 课后练习：为 Exp 加入 tiling 分核

1. N=524288、blocks=8、tile=8192 时，每核负责多少项、循环多少轮、a 与 c 占用多少 UB？写出 bx=3、it=1 的读写区间。
2. 给定 N=1179648、blocks=8，在 <a href="./src/exercises/03.03_tiled_exp.py">Exp 练习骨架</a> 中选择 tile 并补全核起点与核内偏移，运行完整参考比较。
3. 如果 N 不能被 blocks 或 tile 整除，哪些核或哪些轮次需要特殊处理？解释为什么读入、计算、写回都需要考虑有效范围。

完整答案保留 ReLU 的分核、分块索引与搬运，只将计算换为 `T.simd.vexp(ra, mask)`。

<details>
<summary>展开练习解析</summary>

1. 每核 65536 项，8 轮；a、c 合计 `2*8192*4=65536` 字节，即 64 KiB。bx=3、it=1：`begin=3*65536+8192=204800`，区间 [204800,212992)。

2. 例如 tile=1024：每核 147456 项、144 轮；`begin=bx*(N//blocks)+it*tile`。完整答案检查四类输入。

3. 分核可采用商与余数分配；核内末轮按剩余项数确定有效长度。仅增加循环次数仍可能读取或写入界外地址，计算也不能把无效元素当有效输入。边界读写和计算需依据后端能力实现，再检查非整除输入；当前完整答案仅实现正文的整除范围。

</details>

In [2]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)
# 独立子进程使用当前内核的解释器；非零返回码会中止本页。
command = [sys.executable, 'answer/03.03_tiled_exp.py']
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as error:
    print(error.stdout)
    print(error.stderr)
    raise
log_name = Path(command[1]).stem + ("_" + command[2] if len(command)>2 else "") + "_notebook.log"
Path(OUTPUT_DIR / "validation", log_name).write_text(result.stdout + result.stderr)
# 输出节选：省略编译日志与安装路径；完整运行日志写入临时输出目录。
print("\n".join(line for line in result.stdout.splitlines()
                if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

Exp exercise N=1179648 tile=1024 blocks=8 random: full-output PASS; shape=(1179648,); max_abs_error=0
Exp exercise N=1179648 tile=1024 blocks=8 zero: full-output PASS; shape=(1179648,); max_abs_error=0
Exp exercise N=1179648 tile=1024 blocks=8 constant: full-output PASS; shape=(1179648,); max_abs_error=0
Exp exercise N=1179648 tile=1024 blocks=8 ramp: full-output PASS; shape=(1179648,); max_abs_error=0
Verification passed!


[本章导航](README.md) · [上一节](03.02_single_vector.ipynb) · [下一节](03.04_pipeline.ipynb)